In [1]:
import os
import glob
import subprocess
import shutil
import gc
from pathlib import Path
import time
import numpy as np
import librosa
import cv2
import json
import signal
import sys
import psutil

# Global variables
should_stop = False
total_processed = 0

def signal_handler(signum, frame):
    """Handle keyboard interrupt gracefully"""
    global should_stop
    if not should_stop:
        print(f"\n⚠️ Received interrupt signal. Finishing current batch and creating archive...")
        should_stop = True

def save_progress(output_dir, batch_num, total_processed, failed_files):
    """Save progress to resume later"""
    progress_file = os.path.join(output_dir, 'progress.json')
    progress_data = {
        'last_batch': batch_num,
        'total_processed': total_processed,
        'failed_files': failed_files,
        'timestamp': time.time()
    }
    with open(progress_file, 'w') as f:
        json.dump(progress_data, f, indent=2)

def load_progress(output_dir):
    """Load previous progress if exists"""
    progress_file = os.path.join(output_dir, 'progress.json')
    if os.path.exists(progress_file):
        with open(progress_file, 'r') as f:
            return json.load(f)
    return None

def extract_features_batch(mp4_files, output_dir, batch_num, input_dir):
    """Extract features from batch of MP4 files"""
    global should_stop, total_processed
    
    success_count = 0
    failed_files = []
    
    print(f"🔄 Processing batch {batch_num}: {len(mp4_files)} files")
    
    for i, mp4_file in enumerate(mp4_files):
        if should_stop:
            print(f"⚠️ Stopping batch {batch_num} at file {i+1}/{len(mp4_files)}")
            break
            
        try:
            # Build relative path for output
            rel_path = os.path.relpath(mp4_file, input_dir)
            video_output_dir = os.path.join(output_dir, rel_path.replace('.mp4', '').replace('.MP4', ''))
            os.makedirs(video_output_dir, exist_ok=True)
            output_npz = os.path.join(video_output_dir, 'features.npz')
            
            # Skip if already processed
            if os.path.exists(output_npz):
                success_count += 1
                total_processed += 1
                continue
            
            # Extract audio with hardware acceleration if available
            audio_file = os.path.join(video_output_dir, 'temp_audio.wav')
            cmd_audio = [
                'ffmpeg', '-hwaccel', 'auto', '-i', mp4_file,
                '-ar', '16000', '-ac', '1', '-t', '10',
                audio_file, '-hide_banner', '-loglevel', 'error', '-y'
            ]
            subprocess.run(cmd_audio, check=True, timeout=45)
            
            # Extract MFCC (this is CPU-bound, keep it simple)
            y, sr = librosa.load(audio_file, sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=160, win_length=400)
            os.remove(audio_file)
            
            # Extract frames with hardware acceleration
            temp_frames_dir = os.path.join(video_output_dir, 'temp_frames')
            os.makedirs(temp_frames_dir, exist_ok=True)
            cmd_frames = [
                'ffmpeg', '-hwaccel', 'auto', '-i', mp4_file,
                '-vf', 'fps=30,scale=112:112:flags=fast_bilinear,format=gray',
                '-t', '10', '-vframes', '30',
                os.path.join(temp_frames_dir, 'frame_%03d.jpg'),
                '-hide_banner', '-loglevel', 'error', '-y'
            ]
            subprocess.run(cmd_frames, check=True, timeout=45)
            
            # Load frames efficiently
            lip_region = np.zeros((30, 112, 112), dtype=np.uint8)
            for j in range(30):
                frame_path = os.path.join(temp_frames_dir, f'frame_{j+1:03d}.jpg')
                if os.path.exists(frame_path):
                    frame = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
                    if frame is not None:
                        lip_region[j] = frame
            
            # Save features
            np.savez_compressed(output_npz, mfcc=mfcc, lip=lip_region)
            
            # Clean up
            shutil.rmtree(temp_frames_dir)
            
            success_count += 1
            total_processed += 1
            
            # Progress update every 50 files
            if (i + 1) % 50 == 0:
                memory_usage = psutil.virtual_memory().percent
                print(f"  📊 Batch {batch_num}: {i + 1}/{len(mp4_files)} files | Memory: {memory_usage:.1f}%")
                
        except subprocess.TimeoutExpired:
            failed_files.append(mp4_file)
            print(f"  ⏰ Timeout: {os.path.basename(mp4_file)}")
            cleanup_temp_files(locals())
        except Exception as e:
            failed_files.append(mp4_file)
            print(f"  ❌ Error: {os.path.basename(mp4_file)} => {str(e)}")
            cleanup_temp_files(locals())
    
    print(f"✅ Batch {batch_num} completed: {success_count} success, {len(failed_files)} failed")
    return success_count, failed_files

def cleanup_temp_files(local_vars):
    """Clean up temporary files"""
    try:
        if 'audio_file' in local_vars and os.path.exists(local_vars['audio_file']):
            os.remove(local_vars['audio_file'])
        if 'temp_frames_dir' in local_vars and os.path.exists(local_vars['temp_frames_dir']):
            shutil.rmtree(local_vars['temp_frames_dir'])
    except:
        pass

def system_cooldown(batch_num, total_batches):
    """Give system time to cool down between batches"""
    memory_usage = psutil.virtual_memory().percent
    cpu_usage = psutil.cpu_percent(interval=1)
    
    print(f"💾 System Status - Memory: {memory_usage:.1f}%, CPU: {cpu_usage:.1f}%")
    
    # Dynamic cooldown based on system load
    if memory_usage > 80 or cpu_usage > 90:
        cooldown_time = 15  # Longer break for high load
        print(f"🔥 High system load, cooling down for {cooldown_time}s...")
    elif memory_usage > 60 or cpu_usage > 70:
        cooldown_time = 8   # Medium break
        print(f"⚡ Medium load, brief cooldown for {cooldown_time}s...")
    else:
        cooldown_time = 3   # Quick break
        print(f"✨ Low load, quick pause for {cooldown_time}s...")
    
    # Force garbage collection
    gc.collect()
    
    # Sleep with progress indicator
    for i in range(cooldown_time):
        if should_stop:
            break
        time.sleep(1)
        if cooldown_time > 5:  # Only show countdown for longer breaks
            print(f"   ⏳ {cooldown_time - i - 1}s remaining...")

def create_archive(output_dir, total_files):
    """Create archive of extracted features"""
    archive_name = f"/kaggle/working/FakeVideo-RealAudio_features_{total_files}_files.zip"
    print(f"📦 Creating archive: {archive_name}")
    
    try:
        # Disable interrupts during archive creation
        original_handler = signal.signal(signal.SIGINT, signal.SIG_IGN)
        
        # Create archive with progress
        shutil.make_archive(archive_name.replace('.zip', ''), 'zip', output_dir)
        
        # Restore signal handler
        signal.signal(signal.SIGINT, original_handler)
        
        print(f"✅ Archive created: {archive_name}")
        return archive_name
    except Exception as e:
        print(f"❌ Failed to create archive: {e}")
        return None

def estimate_time_remaining(total_files, processed_files, start_time):
    """Estimate remaining processing time"""
    if processed_files == 0:
        return "Unknown"
    
    elapsed_time = time.time() - start_time
    avg_time_per_file = elapsed_time / processed_files
    remaining_files = total_files - processed_files
    remaining_time = remaining_files * avg_time_per_file
    
    hours = int(remaining_time // 3600)
    minutes = int((remaining_time % 3600) // 60)
    return f"{hours}h {minutes}m"

def main():
    global should_stop, total_processed
    
    # Set up signal handler
    signal.signal(signal.SIGINT, signal_handler)
    signal.signal(signal.SIGTERM, signal_handler)
    
    # Configuration
    input_dir = '/kaggle/input/fakeavcaleb/FakeAVCeleb_v1.2/FakeVideo-RealAudio'
    output_dir = '/kaggle/working/FakeVideo-RealAudio/features'
    batch_size = 150  # Optimal batch size for balance
    
    # Verify input directory
    if not os.path.exists(input_dir):
        print(f"❌ Input directory not found: {input_dir}")
        return
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all MP4 files
    print(f"🔍 Scanning for MP4 files...")
    mp4_pattern = os.path.join(input_dir, '**/*.[mM][pP]4')
    all_mp4_files = glob.glob(mp4_pattern, recursive=True)
    
    print(f"🎬 Found {len(all_mp4_files)} MP4 files")
    if len(all_mp4_files) == 0:
        print("❌ No MP4 files found.")
        return
    
    # Check for previous progress
    progress = load_progress(output_dir)
    start_batch = 0
    all_failed_files = []
    
    if progress:
        print(f"📂 Resuming from previous session: {progress['total_processed']} files already processed")
        start_batch = progress['last_batch']
        total_processed = progress['total_processed']
        all_failed_files = progress.get('failed_files', [])
    
    # Calculate batches
    total_batches = (len(all_mp4_files) + batch_size - 1) // batch_size
    start_time = time.time()
    
    print(f"🚀 Starting batch processing:")
    print(f"   📊 Total files: {len(all_mp4_files)}")
    print(f"   📦 Batch size: {batch_size}")
    print(f"   🔢 Total batches: {total_batches}")
    print(f"   ⏰ Starting from batch: {start_batch + 1}")
    
    try:
        # Process batches
        for i in range(start_batch * batch_size, len(all_mp4_files), batch_size):
            if should_stop:
                break
            
            batch_files = all_mp4_files[i:i + batch_size]
            current_batch = (i // batch_size) + 1
            
            print(f"\n{'='*60}")
            print(f"🔄 BATCH {current_batch}/{total_batches}")
            print(f"⏰ ETA: {estimate_time_remaining(len(all_mp4_files), total_processed, start_time)}")
            print(f"{'='*60}")
            
            # Process batch
            success_count, failed_files = extract_features_batch(
                batch_files, output_dir, current_batch, input_dir
            )
            
            all_failed_files.extend(failed_files)
            
            # Save progress
            save_progress(output_dir, current_batch, total_processed, all_failed_files)
            
            print(f"📊 Progress: {total_processed}/{len(all_mp4_files)} files completed")
            
            # System cooldown (skip for last batch)
            if current_batch < total_batches and not should_stop:
                system_cooldown(current_batch, total_batches)
    
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")
        should_stop = True
    
    finally:
        # Final summary
        total_time = time.time() - start_time
        print(f"\n{'='*60}")
        print(f"📊 FINAL SUMMARY")
        print(f"{'='*60}")
        print(f"✅ Successfully processed: {total_processed} files")
        print(f"❌ Failed: {len(all_failed_files)} files")
        print(f"⏱️ Total time: {total_time/60:.1f} minutes")
        print(f"🚀 Average rate: {total_processed/total_time:.1f} files/sec")
        
        # Create archive
        if total_processed > 0:
            archive_name = create_archive(output_dir, total_processed)
            if archive_name:
                print(f"📁 Archive ready: {archive_name}")
        
        # Save failed files
        if all_failed_files:
            with open('/kaggle/working/failed_files.txt', 'w') as f:
                for failed in all_failed_files:
                    f.write(failed + '\n')
            print(f"❗ Failed files saved: /kaggle/working/failed_files.txt")
        
        # Clean up progress file
        progress_file = os.path.join(output_dir, 'progress.json')
        if os.path.exists(progress_file):
            os.remove(progress_file)
        
        print(f"🎉 Processing complete! Archive ready for download.")

if __name__ == "__main__":
    main()

🔍 Scanning for MP4 files...
🎬 Found 9709 MP4 files
🚀 Starting batch processing:
   📊 Total files: 9709
   📦 Batch size: 150
   🔢 Total batches: 65
   ⏰ Starting from batch: 1

🔄 BATCH 1/65
⏰ ETA: Unknown
🔄 Processing batch 1: 150 files
  📊 Batch 1: 50/150 files | Memory: 4.7%
  📊 Batch 1: 100/150 files | Memory: 4.7%
  📊 Batch 1: 150/150 files | Memory: 4.8%
✅ Batch 1 completed: 150 success, 0 failed
📊 Progress: 150/9709 files completed
💾 System Status - Memory: 4.8%, CPU: 0.3%
✨ Low load, quick pause for 3s...

🔄 BATCH 2/65
⏰ ETA: 2h 18m
🔄 Processing batch 2: 150 files
  📊 Batch 2: 50/150 files | Memory: 4.7%
  📊 Batch 2: 100/150 files | Memory: 4.7%
  📊 Batch 2: 150/150 files | Memory: 4.7%
✅ Batch 2 completed: 150 success, 0 failed
📊 Progress: 300/9709 files completed
💾 System Status - Memory: 4.7%, CPU: 0.3%
✨ Low load, quick pause for 3s...

🔄 BATCH 3/65
⏰ ETA: 2h 8m
🔄 Processing batch 3: 150 files

⚠️ Received interrupt signal. Finishing current batch and creating archive...
⚠️ S